# AMEX Default Prediction

Interview-ready client behavior modeling project for predicting default risk on the American Express Kaggle dataset.

## Workflow
1. Data loading
2. EDA
3. Preprocessing and feature engineering
4. Baseline modeling (Logistic Regression + Gradient Boosting)
5. Evaluation (AUC, PR-AUC, decile lift)
6. Explainability and segment insights

## 1. Setup
- Primary data source: Google Drive folder `'/content/drive/MyDrive/amex_data_parquet'`.
- Required files:
  - `train_data.parquet`
  - `train_labels.csv`
- Optional one-time helper block in the next cell downloads missing `train_labels.csv`.


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Tuple
import os
import shutil
import subprocess
import gc
import json
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)

RANDOM_STATE = 42
RUN_PROFILE = 'full_training'  # 'full_training' | 'fast_debug' | 'inference_only'

if RUN_PROFILE == 'full_training':
    MAX_TREND_COLS = 2000
    TEST_MAX_TREND_COLS = 2000
    SAMPLE_FRAC = 1.0
    RETRAIN_FEATURES = False
    RETRAIN_MODELS = True
    USE_INCREMENTAL_LGB = True
    CHUNK_ONLY_MODE = True
    LGB_TRAIN_CHUNKS = 48
    LGB_ROUNDS_PER_CHUNK = 16

elif RUN_PROFILE == 'fast_debug':
    MAX_TREND_COLS = 8
    TEST_MAX_TREND_COLS = 6
    SAMPLE_FRAC = 0.2
    RETRAIN_FEATURES = False
    RETRAIN_MODELS = True
    USE_INCREMENTAL_LGB = True
    CHUNK_ONLY_MODE = True
    LGB_TRAIN_CHUNKS = 16
    LGB_ROUNDS_PER_CHUNK = 8

elif RUN_PROFILE == 'inference_only':
    MAX_TREND_COLS = 15
    TEST_MAX_TREND_COLS = 10
    SAMPLE_FRAC = 1.0
    RETRAIN_FEATURES = False
    RETRAIN_MODELS = False
    USE_INCREMENTAL_LGB = True
    CHUNK_ONLY_MODE = True
    LGB_TRAIN_CHUNKS = 48
    LGB_ROUNDS_PER_CHUNK = 16
else:
    raise ValueError(f'Unknown RUN_PROFILE: {RUN_PROFILE}')

USE_GOOGLE_DRIVE_DATA = True
GOOGLE_DRIVE_DIR = Path('/content/drive/MyDrive/amex_data_parquet')
LOCAL_DATA_DIR = Path('../data/raw/amex-default-prediction')

# 4-layer storage layout
RAW_LAYER_DIR = GOOGLE_DRIVE_DIR
FEATURE_LAYER_DIR = GOOGLE_DRIVE_DIR / 'features'
MODEL_LAYER_DIR = GOOGLE_DRIVE_DIR / 'models'
SUBMISSION_LAYER_DIR = GOOGLE_DRIVE_DIR

TRAIN_FEATURE_PATH = FEATURE_LAYER_DIR / 'customer_features_train.parquet'
TEST_FEATURE_PATH = FEATURE_LAYER_DIR / 'customer_features_test.parquet'
TRAIN_CHUNK_DIR = FEATURE_LAYER_DIR / 'train_chunks'
TEST_CHUNK_DIR = FEATURE_LAYER_DIR / 'test_chunks'
FEATURE_META_PATH = FEATURE_LAYER_DIR / 'feature_meta.json'

MODEL_DIR = MODEL_LAYER_DIR
LOGREG_MODEL_PATH = MODEL_DIR / 'logreg_pipeline.joblib'
LGB_MODEL_PATH = MODEL_DIR / 'lightgbm_baseline.pkl'
XGB_MODEL_PATH = MODEL_DIR / 'xgboost_baseline.joblib'
LGB_META_PATH = MODEL_DIR / 'lightgbm_baseline_meta.json'
MODEL_META_PATH = MODEL_DIR / 'model_meta.json'

BASELINE_SUBMISSION_PATH = SUBMISSION_LAYER_DIR / 'baseline_submission.csv'
SUBMISSION_PARTS_DIR = SUBMISSION_LAYER_DIR / 'submission_parts'
SUBMISSION_META_PATH = SUBMISSION_LAYER_DIR / 'submission_meta.json'
SAMPLE_SUBMISSION_PATH = SUBMISSION_LAYER_DIR / 'sample_submission.csv'

DUCKDB_PATH = GOOGLE_DRIVE_DIR / 'amex_feature_store.duckdb'
FEATURE_SCHEMA_VERSION = 'v1'

KAGGLEHUB_DATASET = 'ruchi798/parquet-files-amexdefault-prediction'
KAGGLE_COMPETITION = 'amex-default-prediction'
USE_GPU_FOR_BOOSTING = True
USE_SPARK_FEATURES = False
SPARK_SHUFFLE_PARTITIONS = '120'

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False
if IN_COLAB and USE_GOOGLE_DRIVE_DATA:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'Drive mount skipped/failed: {e}')


def detect_gpu_available() -> bool:
    if not USE_GPU_FOR_BOOSTING:
        return False
    try:
        run = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
        return run.returncode == 0 and bool(run.stdout.strip())
    except Exception:
        return False

GPU_AVAILABLE = detect_gpu_available()

try:
    from pyspark.sql import SparkSession
    HAS_SPARK = True
except Exception:
    HAS_SPARK = False

spark = None
if HAS_SPARK and USE_SPARK_FEATURES:
    try:
        spark = (
            SparkSession.builder
            .appName('amex-default-prediction')
            .config('spark.sql.shuffle.partitions', SPARK_SHUFFLE_PARTITIONS)
            .config('spark.driver.memory', '16g')
            .getOrCreate()
        )
    except Exception as e:
        print(f'Spark init failed, fallback to pandas features: {e}')
        spark = None

try:
    import kagglehub
    HAS_KAGGLEHUB = True
except Exception:
    HAS_KAGGLEHUB = False

try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

try:
    import catboost as cb
    HAS_CATBOOST = True
except Exception:
    HAS_CATBOOST = False

FEATURE_LAYER_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PARTS_DIR.mkdir(parents=True, exist_ok=True)


# Version gate: choose one pipeline to run
TRAIN_VERSION = 'v2'  # choose exactly one: 'v1' | 'v2'
ALLOW_ALL_VERSIONS = False

FULL_RUN_MIN_LABEL_CUSTOMERS = 400_000
FULL_RUN_MIN_FEATURE_CUSTOMERS = 350_000
FULL_RUN_MIN_COVERAGE_RATIO = 0.85
FULL_RUN_REQUIRE_COMPLETE_CHUNKS = True

_valid_versions = {'v1', 'v2', 'v3'}
if TRAIN_VERSION not in _valid_versions and not (ALLOW_ALL_VERSIONS and TRAIN_VERSION == 'all'):
    raise ValueError(f"Invalid TRAIN_VERSION={TRAIN_VERSION}. Use one of {_valid_versions}.")
if TRAIN_VERSION == 'all' and not ALLOW_ALL_VERSIONS:
    raise ValueError('TRAIN_VERSION=all is disabled to prevent mixed-version artifact mismatch. Pick one version.')

def should_run(version: str) -> bool:
    return TRAIN_VERSION == version or (ALLOW_ALL_VERSIONS and TRAIN_VERSION == 'all')

print({
    'train_version': TRAIN_VERSION,
    'run_profile': RUN_PROFILE,
    'gpu_available': GPU_AVAILABLE,
    'duckdb': HAS_DUCKDB,
    'chunk_only_mode': CHUNK_ONLY_MODE,
    'retrain_features': RETRAIN_FEATURES,
    'retrain_models': RETRAIN_MODELS,
    'incremental_lgb': USE_INCREMENTAL_LGB,
    'full_run_min_label_customers': FULL_RUN_MIN_LABEL_CUSTOMERS,
    'full_run_min_feature_customers': FULL_RUN_MIN_FEATURE_CUSTOMERS,
    'full_run_min_coverage_ratio': FULL_RUN_MIN_COVERAGE_RATIO,
    'catboost': HAS_CATBOOST,
})





## 2. Data Loading

## Full Training Run
- Raw layer: keep `train_data.parquet`, `test_data.parquet`, `train_labels.csv` unchanged.
- Feature-store layer: chunked parquet in `/content/drive/MyDrive/amex_data_parquet/features/*_chunks` + `feature_meta.json`.
- Model layer: model files + `model_meta.json` in `/content/drive/MyDrive/amex_data_parquet/models`.
- Submission layer: streamed prediction parts in `/content/drive/MyDrive/amex_data_parquet/submission_parts` merged into `baseline_submission.csv`.


In [ ]:
# Minimal Colab + Google Drive data loading
# If RETRAIN_FEATURES is False and cached feature files exist, skip raw data loading.

from pathlib import Path
import pandas as pd

base = Path('/content/drive/MyDrive/amex_data_parquet')
labels_path = base / 'train_labels.csv'
train_path = base / 'train_data.parquet'

if (not RETRAIN_FEATURES) and TRAIN_FEATURE_PATH.exists() and TEST_FEATURE_PATH.exists():
    raw_df = None
    labels_df = None
    print('Skipping raw train/test load. Using cached feature files:')
    print({'train_features': str(TRAIN_FEATURE_PATH), 'test_features': str(TEST_FEATURE_PATH)})
else:
    # Keep all Kaggle download operations disabled by default.
    if False:
        # Option A: competition files via Kaggle API
        from google.colab import files
        files.upload()  # pick kaggle.json

        !mkdir -p ~/.kaggle
        !cp kaggle.json ~/.kaggle/kaggle.json
        !chmod 600 ~/.kaggle/kaggle.json

        !kaggle competitions download -c amex-default-prediction -f train_labels.csv -p /content/drive/MyDrive/amex_data_parquet
        !kaggle competitions download -c amex-default-prediction -f sample_submission.csv -p /content/drive/MyDrive/amex_data_parquet

        # Option B: parquet dataset via kagglehub
        # import kagglehub
        # ds_path = kagglehub.dataset_download('ruchi798/parquet-files-amexdefault-prediction')
        # print('Downloaded dataset path:', ds_path)

        import os
        print(sorted(os.listdir('/content/drive/MyDrive/amex_data_parquet'))[:20])

    if not train_path.exists():
        raise FileNotFoundError(f'Missing train file: {train_path}')
    if not labels_path.exists():
        raise FileNotFoundError(f'Missing labels file: {labels_path}')

    train = pd.read_parquet(train_path)
    labels = pd.read_csv(labels_path)
    train['S_2'] = pd.to_datetime(train['S_2'])

    raw_df = train
    labels_df = labels

    print(raw_df.shape, labels_df.shape)
    raw_df.head(3)




## 3. EDA (Quick but Decision-Oriented)

In [ ]:
if raw_df is None or labels_df is None:
    print('EDA skipped: raw data not loaded (using cached feature files).')
else:
    print('Raw columns:', len(raw_df.columns))
    print('Unique customers:', raw_df['customer_ID'].nunique())
    print('Date range:', raw_df['S_2'].min(), 'to', raw_df['S_2'].max())
    display(labels_df['target'].value_counts(normalize=True).rename('target_rate'))


In [ ]:
if raw_df is None:
    print('Missingness EDA skipped: raw data not loaded.')
else:
    missing_rate = raw_df.isna().mean().sort_values(ascending=False)
    display(missing_rate.head(15))


In [ ]:
if raw_df is None or labels_df is None:
    print('Distribution plots skipped: raw data not loaded.')
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    labels_df['target'].value_counts().sort_index().plot(kind='bar', ax=ax[0], title='Target Count (0=Non-default, 1=Default)')
    ax[0].set_xlabel('target')
    obs_per_customer = raw_df.groupby('customer_ID').size()
    obs_per_customer.plot(kind='hist', bins=20, ax=ax[1], title='Statements per customer')
    ax[1].set_xlabel('num_statements')
    plt.tight_layout()


In [ ]:
def safe_slope(values: np.ndarray) -> float:
    mask = ~np.isnan(values)
    if mask.sum() < 2:
        return np.nan
    y = values[mask]
    x = np.arange(len(values))[mask]
    x_centered = x - x.mean()
    denom = np.sum(x_centered ** 2)
    if denom == 0:
        return 0.0
    return np.sum(x_centered * (y - y.mean())) / denom


def make_customer_features_pandas(df: pd.DataFrame, max_trend_cols: int = 40) -> pd.DataFrame:
    df = df.sort_values(['customer_ID', 'S_2']).copy()

    id_cols = ['customer_ID', 'S_2']
    feature_cols = [c for c in df.columns if c not in id_cols]
    numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df[c])]
    trend_cols = numeric_cols[:max_trend_cols]

    grouped = df.groupby('customer_ID', sort=False)
    pieces = []

    num_agg = grouped[numeric_cols].agg(['last', 'mean', 'std', 'min', 'max'])
    num_agg.columns = [f'{col}_{stat}' for col, stat in num_agg.columns]
    pieces.append(num_agg)

    group_size = grouped.size().rename('n_rows')
    num_count = grouped[numeric_cols].count()
    miss = 1.0 - num_count.div(group_size, axis=0)
    miss.columns = [f'{c}_missing_rate' for c in miss.columns]
    pieces.append(miss)

    trend_frames = []
    for c in trend_cols:
        slopes = grouped[c].apply(lambda s: safe_slope(s.values.astype(float)))
        trend_frames.append(slopes.rename(f'{c}_trend'))
    if trend_frames:
        pieces.append(pd.concat(trend_frames, axis=1))

    max_date = df['S_2'].max()
    time_feats = grouped['S_2'].agg(last_statement_date='max', first_date='min', n_statements='count')
    time_feats['recency_days'] = (max_date - time_feats['last_statement_date']).dt.days
    time_feats['history_days'] = (time_feats['last_statement_date'] - time_feats['first_date']).dt.days
    pieces.append(time_feats.drop(columns=['first_date']))

    known_cat_cols = ['B_30', 'B_38', 'D_114', 'D_116', 'D_117', 'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']
    cat_cols_local = [c for c in known_cat_cols if c in df.columns]
    if cat_cols_local:
        cat_last = grouped[cat_cols_local].last().add_suffix('_last_cat')
        cat_nunique = grouped[cat_cols_local].nunique(dropna=True).add_suffix('_nunique_cat')
        pieces.extend([cat_last, cat_nunique])

    return pd.concat(pieces, axis=1).reset_index()


def make_customer_features(df: pd.DataFrame, max_trend_cols: int = 40) -> pd.DataFrame:
    return make_customer_features_pandas(df, max_trend_cols=max_trend_cols)


N_CHUNKS_TRAIN = 24
N_CHUNKS_TEST = 32


def stable_bucket_ids(customer_series: pd.Series, n_chunks: int) -> pd.Series:
    h = pd.util.hash_pandas_object(customer_series.astype('string'), index=False).astype('uint64')
    return (h % np.uint64(n_chunks)).astype('int64')


def build_feature_chunks_to_disk(df: pd.DataFrame, out_dir: Path, n_chunks: int, max_trend_cols: int, prefix: str, retrain: bool) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    if retrain:
        for old in out_dir.glob(f'{prefix}_chunk_*.parquet'):
            old.unlink()

    bucket = stable_bucket_ids(df['customer_ID'], n_chunks)
    chunk_paths = []
    for i in tqdm(range(n_chunks), desc=f'Building {prefix} chunks'):
        cp = out_dir / f'{prefix}_chunk_{i:03d}.parquet'
        chunk_paths.append(cp)
        if cp.exists() and not retrain:
            continue

        part = df.loc[bucket == i].copy()
        if part.empty:
            pd.DataFrame({'customer_ID': []}).to_parquet(cp, index=False)
            del part
            gc.collect()
            continue

        feat = make_customer_features(part, max_trend_cols=max_trend_cols)
        feat.to_parquet(cp, index=False, compression='snappy')
        del part, feat
        gc.collect()
    return chunk_paths


def list_chunk_paths(out_dir: Path, prefix: str) -> list[Path]:
    return sorted(out_dir.glob(f'{prefix}_chunk_*.parquet'))


def infer_feature_schema(chunk_paths: list[Path]) -> tuple[list[str], list[str]]:
    col_union = set()
    cat_union = set()
    for cp in tqdm(chunk_paths, desc='Inferring feature schema from chunks'):
        tmp = pd.read_parquet(cp)
        if tmp.empty:
            continue
        for c in tmp.columns:
            col_union.add(c)
        for c in tmp.columns:
            if str(tmp[c].dtype) in ('object', 'category'):
                cat_union.add(c)
        del tmp
    exclude = {'customer_ID', 'last_statement_date', 'target'}
    feature_cols = sorted([c for c in col_union if c not in exclude])
    cat_cols = sorted([c for c in cat_union if c in feature_cols])
    return feature_cols, cat_cols


def compute_split_cutoff_from_chunks(train_chunk_paths: list[Path], q: float = 0.8) -> pd.Timestamp:
    all_dates = []
    for cp in tqdm(train_chunk_paths, desc='Reading train chunk dates'):
        tmp = pd.read_parquet(cp, columns=['last_statement_date'])
        if not tmp.empty:
            all_dates.append(pd.to_datetime(tmp['last_statement_date']))
        del tmp
    if not all_dates:
        raise RuntimeError('No last_statement_date found in train chunks.')
    cutoff = pd.concat(all_dates, ignore_index=True).quantile(q)
    del all_dates
    gc.collect()
    return pd.to_datetime(cutoff)




def count_rows_from_parquet_footers(paths: list[Path]) -> int:
    total = 0
    try:
        import pyarrow.parquet as pq
        for p in paths:
            if p.exists():
                total += pq.ParquetFile(p).metadata.num_rows
        return int(total)
    except Exception:
        # Fallback if pyarrow metadata read is unavailable
        for p in paths:
            if not p.exists():
                continue
            tmp = pd.read_parquet(p, columns=['customer_ID'])
            total += len(tmp)
            del tmp
            gc.collect()
        return int(total)


def run_full_cache_health_check(
    run_profile: str,
    raw_layer_dir: Path,
    train_chunk_paths: list[Path],
    n_chunks_train: int,
    split_cutoff_date: pd.Timestamp,
    full_run_min_label_customers: int,
    full_run_min_feature_customers: int,
    full_run_min_coverage_ratio: float,
    require_complete_chunks: bool,
) -> dict:
    report = {
        'run_profile': run_profile,
        'split_cutoff_date': str(split_cutoff_date),
        'train_chunk_count_found': len(train_chunk_paths),
        'train_chunk_count_expected': n_chunks_train,
    }

    labels_path = raw_layer_dir / 'train_labels.csv'
    if labels_path.exists():
        n_labels = int(pd.read_csv(labels_path, usecols=['customer_ID']).shape[0])
    else:
        n_labels = 0
    n_feature_rows = count_rows_from_parquet_footers(train_chunk_paths)

    coverage = (n_feature_rows / n_labels) if n_labels else 0.0
    report.update({
        'label_customers': n_labels,
        'feature_customers': n_feature_rows,
        'coverage_ratio': round(float(coverage), 6),
    })

    if run_profile == 'full_training':
        if require_complete_chunks and len(train_chunk_paths) != n_chunks_train:
            raise RuntimeError(
                f'Cache health check failed: expected {n_chunks_train} train chunks, found {len(train_chunk_paths)}. '
                'Set RETRAIN_FEATURES=True and rebuild feature chunks.'
            )

        if n_labels < full_run_min_label_customers:
            raise RuntimeError(
                f'Cache health check failed: labels too small for full run ({n_labels} < {full_run_min_label_customers}). '
                'Verify RAW_LAYER_DIR points to full Kaggle dataset.'
            )

        if n_feature_rows < full_run_min_feature_customers:
            raise RuntimeError(
                f'Cache health check failed: feature customers too small for full run '
                f'({n_feature_rows} < {full_run_min_feature_customers}). '
                'Likely stale cache from debug/sample run. Set RETRAIN_FEATURES=True once.'
            )

        if coverage < full_run_min_coverage_ratio:
            raise RuntimeError(
                f'Cache health check failed: feature coverage too low ({coverage:.3f} < {full_run_min_coverage_ratio:.3f}). '
                'Rebuild feature chunks from full raw data.'
            )

    return report

def sync_duckdb_feature_store(train_chunk_paths: list[Path], test_chunk_paths: list[Path], feature_meta: dict) -> None:
    if not HAS_DUCKDB:
        return
    con = duckdb.connect(str(DUCKDB_PATH))
    train_glob = str(TRAIN_CHUNK_DIR / 'train_chunk_*.parquet')
    test_glob = str(TEST_CHUNK_DIR / 'test_chunk_*.parquet')
    con.execute(f"CREATE OR REPLACE VIEW features_train_chunks AS SELECT * FROM read_parquet('{train_glob}')")
    con.execute(f"CREATE OR REPLACE VIEW features_test_chunks AS SELECT * FROM read_parquet('{test_glob}')")
    con.execute('CREATE OR REPLACE TABLE feature_meta AS SELECT ? AS meta_json', [json.dumps(feature_meta)])
    con.close()


# Build/list train chunks
if raw_df is not None:
    train_chunk_paths = build_feature_chunks_to_disk(
        raw_df,
        out_dir=TRAIN_CHUNK_DIR,
        n_chunks=N_CHUNKS_TRAIN,
        max_trend_cols=MAX_TREND_COLS,
        prefix='train',
        retrain=RETRAIN_FEATURES,
    )
else:
    train_chunk_paths = list_chunk_paths(TRAIN_CHUNK_DIR, 'train')

if not train_chunk_paths:
    raise RuntimeError('No train chunks found. Enable raw loading once to build train chunks.')

expected_train = [TRAIN_CHUNK_DIR / f'train_chunk_{i:03d}.parquet' for i in range(N_CHUNKS_TRAIN)]
missing_train = [cp for cp in expected_train if cp not in set(train_chunk_paths)]
if missing_train and raw_df is not None and (not RETRAIN_FEATURES):
    print(f'Resuming train chunk build: {len(train_chunk_paths)} done, {len(missing_train)} missing.')
    train_chunk_paths = build_feature_chunks_to_disk(
        raw_df,
        out_dir=TRAIN_CHUNK_DIR,
        n_chunks=N_CHUNKS_TRAIN,
        max_trend_cols=MAX_TREND_COLS,
        prefix='train',
        retrain=False,
    )
elif missing_train and raw_df is None and (not RETRAIN_FEATURES):
    print(f'Warning: train chunks incomplete ({len(train_chunk_paths)}/{N_CHUNKS_TRAIN}) and raw_df not loaded.')

# Build/list test chunks when raw test is available.
# Explicit local variable avoids cell-order ambiguity.
test_raw_local = globals().get('test_raw', None)
if test_raw_local is not None:
    test_chunk_paths = build_feature_chunks_to_disk(
        test_raw_local,
        out_dir=TEST_CHUNK_DIR,
        n_chunks=N_CHUNKS_TEST,
        max_trend_cols=TEST_MAX_TREND_COLS,
        prefix='test',
        retrain=RETRAIN_FEATURES,
    )
else:
    test_chunk_paths = list_chunk_paths(TEST_CHUNK_DIR, 'test')

# Feature metadata
if FEATURE_META_PATH.exists() and not RETRAIN_FEATURES:
    feature_meta = json.loads(FEATURE_META_PATH.read_text())
    print(f'Loaded feature_meta: {FEATURE_META_PATH}')
else:
    feature_cols_meta, cat_cols_meta = infer_feature_schema(train_chunk_paths)
    split_cutoff_meta = str(compute_split_cutoff_from_chunks(train_chunk_paths, q=0.8))

    feature_meta = {
        'schema_version': FEATURE_SCHEMA_VERSION,
        'created_at': datetime.utcnow().isoformat(),
        'train_chunk_count': len(train_chunk_paths),
        'test_chunk_count': len(test_chunk_paths),
        'feature_columns': feature_cols_meta,
        'categorical_columns': cat_cols_meta,
        'split_cutoff_date': split_cutoff_meta,
        'max_trend_cols_train': MAX_TREND_COLS,
        'max_trend_cols_test': TEST_MAX_TREND_COLS,
    }
    FEATURE_META_PATH.write_text(json.dumps(feature_meta, indent=2))
    print(f'Saved feature_meta: {FEATURE_META_PATH}')

sync_duckdb_feature_store(train_chunk_paths, test_chunk_paths, feature_meta)

# Chunk-only mode: no full train concat
customer_features = None
feature_cols = feature_meta['feature_columns']
cat_cols = feature_meta.get('categorical_columns', [])
split_cutoff_date = pd.to_datetime(feature_meta['split_cutoff_date'])

cache_health = run_full_cache_health_check(
    run_profile=RUN_PROFILE,
    raw_layer_dir=RAW_LAYER_DIR,
    train_chunk_paths=train_chunk_paths,
    n_chunks_train=N_CHUNKS_TRAIN,
    split_cutoff_date=split_cutoff_date,
    full_run_min_label_customers=FULL_RUN_MIN_LABEL_CUSTOMERS,
    full_run_min_feature_customers=FULL_RUN_MIN_FEATURE_CUSTOMERS,
    full_run_min_coverage_ratio=FULL_RUN_MIN_COVERAGE_RATIO,
    require_complete_chunks=FULL_RUN_REQUIRE_COMPLETE_CHUNKS,
)

print({
    'chunk_only_mode': CHUNK_ONLY_MODE,
    'n_train_chunks': len(train_chunk_paths),
    'n_features': len(feature_cols),
    'split_cutoff_date': str(split_cutoff_date),
})
print({'cache_health': cache_health})

In [ ]:
# Deprecated duplicate split cell kept intentionally for backward notebook order.
# Use the next split metadata cell only.
print('Deprecated: skip this cell. Run the next split metadata cell.')

In [ ]:
# Time-aware split metadata
if CHUNK_ONLY_MODE:
    all_dates = []
    for cp in tqdm(train_chunk_paths, desc='Reading train chunk dates'):
        tmp = pd.read_parquet(cp, columns=['last_statement_date'])
        if not tmp.empty:
            all_dates.append(pd.to_datetime(tmp['last_statement_date']))
        del tmp
    if not all_dates:
        raise RuntimeError('No dates found in train chunks.')
    split_cutoff_date = pd.concat(all_dates, ignore_index=True).quantile(0.8)
    del all_dates
    gc.collect()

    model_df = None
    train_df = None
    valid_df = None
    print('Chunk-only mode split cutoff:', split_cutoff_date)
else:
    model_df = customer_features.sort_values('last_statement_date').reset_index(drop=True)
    cut_idx = int(len(model_df) * 0.8)
    train_df = model_df.iloc[:cut_idx].copy()
    valid_df = model_df.iloc[cut_idx:].copy()
    print('Train period max date:', train_df['last_statement_date'].max())
    print('Valid period min date:', valid_df['last_statement_date'].min())
    print('Train size:', train_df.shape, 'Valid size:', valid_df.shape)



## 5. Baseline Modeling

In [ ]:
if not should_run('v1'):
    print("Skipping v1 training (TRAIN_VERSION != 'v1'/'all').")
else:
    # Gradient boosting baseline (LightGBM chunk-incremental, full feature schema)
    TARGET_COL = globals().get('TARGET_COL', 'target')
    EXCLUDE_COLS = globals().get('EXCLUDE_COLS', ['customer_ID', 'last_statement_date', TARGET_COL])

    boost_model_name = None
    boost_device = 'cpu'
    proba_boost = None
    boost_importance = None


    def _prepare_lgb_frame(df: pd.DataFrame, cat_columns: list[str]) -> pd.DataFrame:
        out = df.copy()
        for c in cat_columns:
            if c in out.columns:
                out[c] = out[c].astype('category')
        return out


    def _load_labels_map() -> pd.DataFrame:
        labels_path = RAW_LAYER_DIR / 'train_labels.csv'
        lbl = pd.read_csv(labels_path)
        lbl['customer_ID'] = lbl['customer_ID'].astype('string')
        lbl['target'] = pd.to_numeric(lbl['target'], errors='coerce')
        return lbl[['customer_ID', 'target']]


    if HAS_LGBM:
        boost_model_name = 'lightgbm'
        labels_map = _load_labels_map()

        lgb_params = dict(
            objective='binary',
            learning_rate=0.03,
            num_leaves=64,
            subsample=0.9,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1,
        )
        if GPU_AVAILABLE:
            lgb_params['device'] = 'gpu'

        booster = None
        if LGB_MODEL_PATH.exists() and not RETRAIN_MODELS:
            booster = joblib.load(LGB_MODEL_PATH)
            print(f'Loaded LightGBM model: {LGB_MODEL_PATH}')
        else:
            if not USE_INCREMENTAL_LGB:
                raise RuntimeError('CHUNK_ONLY_MODE requires USE_INCREMENTAL_LGB=True.')

            print({'mode': 'incremental_lgb', 'chunks': len(train_chunk_paths), 'rounds_per_chunk': LGB_ROUNDS_PER_CHUNK, 'n_features': len(feature_cols)})

            try_gpu = GPU_AVAILABLE
            for i, cp in enumerate(train_chunk_paths, start=1):
                chunk_df = pd.read_parquet(cp)
                if chunk_df.empty:
                    continue

                chunk_df['customer_ID'] = chunk_df['customer_ID'].astype('string')
                if 'target' not in chunk_df.columns:
                    chunk_df = chunk_df.merge(labels_map, on='customer_ID', how='left')

                train_part = chunk_df[chunk_df['last_statement_date'] <= split_cutoff_date].copy()
                train_part = train_part[train_part['target'].notna()].copy()
                if train_part.empty:
                    del chunk_df, train_part
                    gc.collect()
                    continue

                for c in feature_cols:
                    if c not in train_part.columns:
                        train_part[c] = np.nan

                x_chunk = _prepare_lgb_frame(train_part[feature_cols], cat_cols)
                y_chunk = train_part['target'].astype(int)

                dtrain = lgb.Dataset(
                    x_chunk,
                    label=y_chunk,
                    categorical_feature=[c for c in cat_cols if c in x_chunk.columns],
                    free_raw_data=True,
                )

                try:
                    booster = lgb.train(
                        lgb_params,
                        dtrain,
                        num_boost_round=LGB_ROUNDS_PER_CHUNK,
                        init_model=booster,
                        keep_training_booster=True,
                    )
                    boost_device = 'gpu' if try_gpu else 'cpu'
                except Exception as e:
                    if try_gpu:
                        print(f'LightGBM GPU failed in chunk {i}, fallback to CPU: {e}')
                        lgb_params.pop('device', None)
                        try_gpu = False
                        booster = lgb.train(
                            lgb_params,
                            dtrain,
                            num_boost_round=LGB_ROUNDS_PER_CHUNK,
                            init_model=booster,
                            keep_training_booster=True,
                        )
                        boost_device = 'cpu'
                    else:
                        raise

                del chunk_df, train_part, x_chunk, y_chunk, dtrain
                gc.collect()
                if i % 2 == 0 or i == len(train_chunk_paths):
                    print(f'Finished train chunk {i}/{len(train_chunk_paths)}')

            if booster is None:
                raise RuntimeError('Incremental training produced no booster.')

            joblib.dump(booster, LGB_MODEL_PATH)
            print(f'Saved LightGBM model: {LGB_MODEL_PATH}')

        gbm = booster

        # Validation scoring streamed from train chunks
        y_valid_list = []
        pred_valid_list = []
        X_valid_sample_parts = []

        for cp in tqdm(train_chunk_paths, desc='Scoring valid chunks'):
            chunk_df = pd.read_parquet(cp)
            if chunk_df.empty:
                continue

            chunk_df['customer_ID'] = chunk_df['customer_ID'].astype('string')
            if 'target' not in chunk_df.columns:
                chunk_df = chunk_df.merge(labels_map, on='customer_ID', how='left')

            valid_part = chunk_df[chunk_df['last_statement_date'] > split_cutoff_date].copy()
            valid_part = valid_part[valid_part['target'].notna()].copy()
            if valid_part.empty:
                del chunk_df, valid_part
                gc.collect()
                continue

            for c in feature_cols:
                if c not in valid_part.columns:
                    valid_part[c] = np.nan

            x_valid_chunk = _prepare_lgb_frame(valid_part[feature_cols], cat_cols)
            y_valid_list.append(valid_part['target'].astype(int).values)
            pred_valid_list.append(gbm.predict(x_valid_chunk))

            if len(X_valid_sample_parts) < 4:
                X_valid_sample_parts.append(x_valid_chunk.head(500))

            del chunk_df, valid_part, x_valid_chunk
            gc.collect()

        if not y_valid_list:
            raise RuntimeError('No validation rows found across chunks.')

        y_valid = pd.Series(np.concatenate(y_valid_list), name='target')
        proba_boost = np.concatenate(pred_valid_list)

        valid_mask = y_valid.notna() & pd.Series(proba_boost).notna()
        dropped = int((~valid_mask).sum())
        if dropped > 0:
            print(f'Dropping {dropped} rows with NaN before metrics.')
        y_valid = y_valid[valid_mask].astype(int).reset_index(drop=True)
        proba_boost = np.asarray(proba_boost)[valid_mask.values]

        X_valid = pd.concat(X_valid_sample_parts, axis=0, ignore_index=True) if X_valid_sample_parts else None

        boost_importance = pd.Series(gbm.feature_importance(), index=gbm.feature_name())
        auc_boost = roc_auc_score(y_valid, proba_boost)
        pr_auc_boost = average_precision_score(y_valid, proba_boost)

        model_meta = {
            'created_at': datetime.utcnow().isoformat(),
            'model_name': 'lightgbm_incremental',
            'model_path': str(LGB_MODEL_PATH),
            'feature_columns': feature_cols,
            'categorical_columns': cat_cols,
            'split_cutoff_date': str(split_cutoff_date),
            'auc': float(auc_boost),
            'pr_auc': float(pr_auc_boost),
            'feature_schema_version': feature_meta.get('schema_version', FEATURE_SCHEMA_VERSION),
        }
        MODEL_META_PATH.write_text(json.dumps(model_meta, indent=2))
        LGB_META_PATH.write_text(json.dumps(model_meta, indent=2))

        print({'model': 'lightgbm', 'device': boost_device, 'auc': round(auc_boost, 5), 'pr_auc': round(pr_auc_boost, 5)})
    else:
        raise RuntimeError('LightGBM is required for chunk-only full training flow.')





In [ ]:
# Deprecated duplicate baseline training cell.
# Use the previous v1 training cell only.
print('Deprecated: duplicate baseline cell disabled. Use the previous v1 training cell.')


In [ ]:
def decile_lift_table(y_true: pd.Series, y_score: np.ndarray) -> pd.DataFrame:
    eval_df = pd.DataFrame({'y_true': y_true.values, 'score': y_score})
    eval_df['decile'] = pd.qcut(eval_df['score'].rank(method='first'), 10, labels=False) + 1
    eval_df['decile'] = 11 - eval_df['decile']

    base_rate = eval_df['y_true'].mean()
    out = (
        eval_df.groupby('decile')
        .agg(n=('y_true', 'size'), bad_rate=('y_true', 'mean'))
        .reset_index()
        .sort_values('decile')
    )
    out['lift_vs_avg'] = out['bad_rate'] / base_rate
    return out

results = [{'model': 'lightgbm', 'auc': auc_boost, 'pr_auc': pr_auc_boost}]
results_df = pd.DataFrame(results)
results_df



In [ ]:
lift_boost = decile_lift_table(y_valid, proba_boost)
print('LightGBM decile lift:')
display(lift_boost)

plt.figure(figsize=(8, 4))
plt.plot(lift_boost['decile'], lift_boost['lift_vs_avg'], marker='o', label='lightgbm')
plt.axhline(1.0, linestyle='--', color='gray')
plt.title('Lift by Risk Decile (1 = Highest Predicted Risk)')
plt.xlabel('Decile')
plt.ylabel('Lift vs Average')
plt.legend()
plt.tight_layout()



In [ ]:
proba_lr_local = globals().get('proba_lr', None)
proba_boost_local = globals().get('proba_boost', None)
boost_model_name_local = globals().get('boost_model_name', 'boost_model')
y_valid_local = globals().get('y_valid', None)

if y_valid_local is None:
    raise RuntimeError('y_valid is missing. Run model training/evaluation cells first.')

if proba_lr_local is not None:
    lift_lr = decile_lift_table(y_valid_local, proba_lr_local)
    print('Logistic regression decile lift:')
    display(lift_lr)
else:
    lift_lr = None

if proba_boost_local is not None:
    lift_boost = decile_lift_table(y_valid_local, proba_boost_local)
    print(f'{boost_model_name_local} decile lift:')
    display(lift_boost)
else:
    lift_boost = None

plt.figure(figsize=(8, 4))
if lift_lr is not None:
    plt.plot(lift_lr['decile'], lift_lr['lift_vs_avg'], marker='o', label='Logistic Regression')
if lift_boost is not None:
    plt.plot(lift_boost['decile'], lift_boost['lift_vs_avg'], marker='o', label=boost_model_name_local)
plt.axhline(1.0, linestyle='--', color='gray')
plt.title('Lift by Risk Decile (1 = Highest Predicted Risk)')
plt.xlabel('Decile')
plt.ylabel('Lift vs Average')
if (lift_lr is not None) or (lift_boost is not None):
    plt.legend()
plt.tight_layout()



## 7. Explainability

In [ ]:
if boost_importance is not None:
    top_imp = boost_importance.sort_values(ascending=False).head(20)
    plt.figure(figsize=(8, 6))
    sns.barplot(x=top_imp.values, y=top_imp.index, orient='h')
    plt.title(f'Top 20 Global Feature Importances ({boost_model_name})')
    plt.tight_layout()

    display(top_imp.to_frame('importance'))

In [ ]:
if HAS_SHAP and proba_boost is not None and 'X_valid' in globals() and X_valid is not None and len(X_valid) > 0:
    shap_sample = X_valid.sample(n=min(2000, len(X_valid)), random_state=RANDOM_STATE).copy()

    if boost_model_name == 'lightgbm':
        for c in cat_cols:
            if c in shap_sample.columns:
                shap_sample[c] = shap_sample[c].astype('category')
        explainer = shap.TreeExplainer(gbm)
        shap_values = explainer.shap_values(shap_sample)
        shap.summary_plot(shap_values, shap_sample, show=True)

    elif boost_model_name == 'xgboost':
        shap_sample_enc = shap_sample.copy()
        for c in cat_cols:
            shap_sample_enc[c] = shap_sample_enc[c].astype('category').cat.codes
        explainer = shap.TreeExplainer(xgb_model)
        shap_values = explainer.shap_values(shap_sample_enc)
        shap.summary_plot(shap_values, shap_sample_enc, show=True)
else:
    print('SHAP skipped (missing package/model or no in-memory validation sample).')



print('Segment summary skipped in chunk-only/database-style run. Use sampled valid table if needed.')


In [ ]:
valid_df_local = globals().get('valid_df', None)
chunk_mode_local = globals().get('CHUNK_ONLY_MODE', False)
proba_lr_local = globals().get('proba_lr', None)
proba_boost_local = globals().get('proba_boost', None)

if chunk_mode_local or (valid_df_local is None):
    print('Segment summary skipped in chunk-only mode or missing valid_df.')
else:
    analysis_df = valid_df_local[['customer_ID', 'target']].copy()

    if proba_lr_local is not None:
        analysis_df['score'] = proba_lr_local
        score_name = 'logistic_regression'
    elif proba_boost_local is not None:
        analysis_df['score'] = proba_boost_local
        score_name = globals().get('boost_model_name', 'boost_model')
    else:
        raise RuntimeError('No model scores found. Run a model prediction cell first.')

    analysis_df['risk_decile'] = pd.qcut(analysis_df['score'].rank(method='first'), 10, labels=False) + 1
    analysis_df['risk_decile'] = 11 - analysis_df['risk_decile']

    key_behavior_cols = [
        c for c in valid_df_local.columns
        if c.endswith('_trend') or c.endswith('_std') or c.endswith('_missing_rate')
    ]
    key_behavior_cols = key_behavior_cols[:12]

    segment_view = valid_df_local[['customer_ID'] + key_behavior_cols].merge(
        analysis_df[['customer_ID', 'risk_decile', 'target']],
        on='customer_ID', how='left'
    )

    segment_summary = (
        segment_view.groupby('risk_decile')[key_behavior_cols + ['target']]
        .mean()
        .sort_index()
    )

    print(f'Segment summary based on: {score_name}')
    display(segment_summary.head(10))



## 9. Interview Talking Points

- Why time-aware validation matters: random splits can leak future behavior into training; customer last-statement split is safer.
- Feature design: recency, volatility (`std`), directional trend (`trend`), and missingness patterns often carry default signals.
- Baseline progression: start linear for interpretability, then use gradient boosting to capture nonlinearity and interactions.
- Business usage: decile lift supports risk-tier policying (collections prioritization, credit line review, proactive outreach).
- Next improvements: target encoding with out-of-fold strategy, sequence models, and monotonic constraints for risk governance.

## 10. Load Test Data Only

Load test parquet once, then reuse `test_raw` in submission cells.


In [ ]:
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/amex_data_parquet')
TEST_PATH = BASE_DIR / 'test_data.parquet'

if (not RETRAIN_FEATURES) and TEST_FEATURE_PATH.exists():
    test_raw = None
    print(f'Skipping raw test load. Using cached test features: {TEST_FEATURE_PATH}')
else:
    if not TEST_PATH.exists():
        raise FileNotFoundError(f'Missing test file: {TEST_PATH}')
    test_raw = pd.read_parquet(TEST_PATH)
    test_raw['S_2'] = pd.to_datetime(test_raw['S_2'])
    print('Loaded test_raw:', test_raw.shape)
    test_raw.head(3)



In [ ]:
def ensure_test_chunks(test_raw: pd.DataFrame | None = None) -> list[Path]:
    chunk_paths = list_chunk_paths(TEST_CHUNK_DIR, 'test')

    expected = [TEST_CHUNK_DIR / f'test_chunk_{i:03d}.parquet' for i in range(N_CHUNKS_TEST)]
    existing_set = set(chunk_paths)
    missing = [cp for cp in expected if cp not in existing_set]

    if chunk_paths and not missing and (not RETRAIN_FEATURES):
        print(f'Using existing complete test chunks: {len(chunk_paths)}/{N_CHUNKS_TEST}')
        return chunk_paths

    if chunk_paths and missing and (not RETRAIN_FEATURES):
        print(f'Resuming test chunk build: {len(chunk_paths)} done, {len(missing)} missing.')
        print('First missing chunk:', missing[0].name)

    if test_raw is None:
        if missing:
            raise RuntimeError(
                f'Test chunks are incomplete ({len(chunk_paths)}/{N_CHUNKS_TEST}). '
                'Load test_raw once to continue building unfinished chunks.'
            )
        raise RuntimeError('Raw test data is required to build test chunks once.')

    # build_feature_chunks_to_disk skips existing chunk files when retrain=False,
    # so this call naturally continues from unfinished chunks only.
    return build_feature_chunks_to_disk(
        test_raw,
        out_dir=TEST_CHUNK_DIR,
        n_chunks=N_CHUNKS_TEST,
        max_trend_cols=TEST_MAX_TREND_COLS,
        prefix='test',
        retrain=RETRAIN_FEATURES,
    )



In [ ]:
if not should_run('v1'):
    print("Skipping v1 submission (TRAIN_VERSION != 'v1'/'all').")
else:
    # Streamed baseline submission generation from test chunks (memory-safe)

    test_chunk_paths = ensure_test_chunks(globals().get('test_raw', None))

    if MODEL_META_PATH.exists():
        model_meta = json.loads(MODEL_META_PATH.read_text())
    else:
        raise RuntimeError('Missing model_meta.json. Train/load model first.')

    feature_cols = model_meta['feature_columns']
    cat_cols = model_meta.get('categorical_columns', [])

    for old in SUBMISSION_PARTS_DIR.glob('part_*.csv'):
        old.unlink()

    model_used = 'lightgbm'
    for i, cp in enumerate(tqdm(test_chunk_paths, desc='Scoring test chunks')):
        chunk_df = pd.read_parquet(cp)
        if chunk_df.empty:
            continue

        for c in feature_cols:
            if c not in chunk_df.columns:
                chunk_df[c] = np.nan

        x_chunk = _prepare_lgb_frame(chunk_df[feature_cols], cat_cols)
        pred = gbm.predict(x_chunk) if isinstance(gbm, lgb.Booster) else gbm.predict_proba(x_chunk)[:, 1]

        pd.DataFrame({'customer_ID': chunk_df['customer_ID'], 'prediction': pred}).to_csv(
            SUBMISSION_PARTS_DIR / f'part_{i:03d}.csv', index=False
        )

        del chunk_df, x_chunk, pred
        gc.collect()

    submission = pd.concat(
    [pd.read_csv(x) for x in sorted(SUBMISSION_PARTS_DIR.glob('part_*.csv'))],
    axis=0,
    ignore_index=True,
)

# Enforce Kaggle-required id set/order
if SAMPLE_SUBMISSION_PATH.exists():
    sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    submission = submission.drop_duplicates(subset=['customer_ID'], keep='last')
    submission = sample_sub[['customer_ID']].merge(submission[['customer_ID', 'prediction']], on='customer_ID', how='left')
    if submission['prediction'].isna().any():
        fill_val = submission['prediction'].median(skipna=True)
        if pd.isna(fill_val):
            fill_val = 0.5
        submission['prediction'] = submission['prediction'].fillna(fill_val)
else:
    print(f'Warning: sample submission file not found at {SAMPLE_SUBMISSION_PATH}. Saving raw submission order.')

submission.to_csv(BASELINE_SUBMISSION_PATH, index=False)

submission_meta = {
    'created_at': datetime.utcnow().isoformat(),
    'model_used': model_used,
    'submission_path': str(BASELINE_SUBMISSION_PATH),
    'rows': int(len(submission)),
    'unique_customer_ids': int(submission['customer_ID'].nunique()),
    'null_predictions': int(submission['prediction'].isna().sum()),
    'parts': len(list(SUBMISSION_PARTS_DIR.glob('part_*.csv'))),
}
SUBMISSION_META_PATH.write_text(json.dumps(submission_meta, indent=2))

print({'model_used': model_used, 'rows': len(submission), 'unique_ids': submission['customer_ID'].nunique(), 'saved_to': str(BASELINE_SUBMISSION_PATH)})
submission.head()



In [ ]:
print('Legacy duplicate submission cell disabled. Use the v1 cell above or v2 advanced cell below.')


In [ ]:
print('Legacy advanced v2 cell disabled. Use the final advanced v2 production cell below.')


In [ ]:
# Deprecated legacy advanced cell.
# Keep for notebook order compatibility only.
print('Deprecated: legacy advanced cell disabled. Use the final Advanced v2 production cell below.')


## 13. Advanced Model v2 (Production Ensemble)

Train a production advanced ensemble with chunk-incremental LightGBM branches plus CatBoost blending (when available), optimized using the AMEX metric, and generate `advanced_submission_v2.csv`.


In [ ]:
if not should_run('v2'):
    print("Skipping advanced v2 pipeline (TRAIN_VERSION != 'v2').")
else:
    # Fallback helpers so this cell can run independently after kernel restart.
    if 'ensure_train_chunks' not in globals():
        def ensure_train_chunks(raw_train: pd.DataFrame | None = None) -> list[Path]:
            paths = list_chunk_paths(TRAIN_CHUNK_DIR, 'train')
            if paths:
                return paths
            if raw_train is None:
                raise RuntimeError('ensure_train_chunks missing and no existing train chunks found.')
            return build_feature_chunks_to_disk(
                raw_train,
                out_dir=TRAIN_CHUNK_DIR,
                n_chunks=N_CHUNKS_TRAIN,
                max_trend_cols=MAX_TREND_COLS,
                prefix='train',
                retrain=RETRAIN_FEATURES,
            )

    if 'ensure_test_chunks' not in globals():
        def ensure_test_chunks(test_raw: pd.DataFrame | None = None) -> list[Path]:
            paths = list_chunk_paths(TEST_CHUNK_DIR, 'test')
            if paths:
                return paths
            if test_raw is None:
                raise RuntimeError('ensure_test_chunks missing and no existing test chunks found.')
            return build_feature_chunks_to_disk(
                test_raw,
                out_dir=TEST_CHUNK_DIR,
                n_chunks=N_CHUNKS_TEST,
                max_trend_cols=TEST_MAX_TREND_COLS,
                prefix='test',
                retrain=RETRAIN_FEATURES,
            )

    ADVANCED_SUBMISSION_V2_PATH = SUBMISSION_LAYER_DIR / 'advanced_submission_v2.csv'
    ADV_V2_DIR = MODEL_DIR / 'advanced_v2'
    ADV_V2_DIR.mkdir(parents=True, exist_ok=True)

    # Defensive: always define chunk paths in this cell.
    train_chunk_paths = ensure_train_chunks(globals().get('raw_df', None))
    test_chunk_paths = ensure_test_chunks(globals().get('test_raw', None))

    N_OOF_FOLDS = 5

    def amex_metric_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)

        order = np.argsort(-y_pred)
        y_true_sorted = y_true[order]
        w = np.where(y_true_sorted == 0, 20, 1)

        cut = int(np.ceil(0.04 * w.sum()))
        top_mask = np.cumsum(w) <= cut
        d = (y_true_sorted[top_mask] == 1).sum() / max((y_true_sorted == 1).sum(), 1)

        def weighted_gini(a_true, a_pred):
            idx = np.argsort(-a_pred)
            a_true = a_true[idx]
            ww = np.where(a_true == 0, 20, 1)
            cum_w = np.cumsum(ww)
            cum_pos = np.cumsum(a_true * ww)
            lorentz = cum_pos / cum_pos[-1]
            g = np.sum((lorentz - cum_w / cum_w[-1]) * ww)
            return g

        g = weighted_gini(y_true, y_pred) / max(weighted_gini(y_true, y_true), 1e-12)
        return 0.5 * (g + d)

    def rank01(x: np.ndarray) -> np.ndarray:
        return pd.Series(x).rank(method='average', pct=True).values

    def get_branch_cols(feature_cols_all: list[str]) -> tuple[list[str], list[str]]:
        cols_a = feature_cols_all[:]
        keep_suffix = ('_last', '_mean', '_std', '_missing_rate', '_last_cat', '_nunique_cat')
        keep_exact = {'recency_days', 'history_days', 'n_statements'}
        cols_b = [c for c in feature_cols_all if c.endswith(keep_suffix) or c in keep_exact]
        cols_b = sorted(list(set(cols_b)))
        if not cols_b:
            cols_b = cols_a[:]
        return cols_a, cols_b

    def add_fold_id(df: pd.DataFrame, n_folds: int) -> pd.Series:
        h = pd.util.hash_pandas_object(df['customer_ID'].astype('string'), index=False).astype('uint64')
        return (h % np.uint64(n_folds)).astype('int64')

    def fit_incremental_booster_for_folds(
        cols_use: list[str],
        cat_cols_all: list[str],
        params: dict,
        rounds_per_chunk: int,
        train_fold_ids: set[int],
    ):
        labels_map = _load_labels_map()
        booster = None
        cat_use = [c for c in cat_cols_all if c in cols_use]

        local_params = params.copy()
        use_gpu = 'device' in local_params and local_params['device'] == 'gpu'

        for cp in train_chunk_paths:
            chunk_df = pd.read_parquet(cp)
            if chunk_df.empty:
                continue

            chunk_df['customer_ID'] = chunk_df['customer_ID'].astype('string')
            if 'target' not in chunk_df.columns:
                chunk_df = chunk_df.merge(labels_map, on='customer_ID', how='left')

            chunk_df = chunk_df[chunk_df['target'].notna()].copy()
            if chunk_df.empty:
                del chunk_df
                gc.collect()
                continue

            chunk_df['fold_id'] = add_fold_id(chunk_df, N_OOF_FOLDS)
            part = chunk_df[chunk_df['fold_id'].isin(train_fold_ids)].copy()
            if part.empty:
                del chunk_df, part
                gc.collect()
                continue

            for c in cols_use:
                if c not in part.columns:
                    part[c] = np.nan

            x = _prepare_lgb_frame(part[cols_use], cat_use)
            y = part['target'].astype(int)

            dtrain = lgb.Dataset(
                x,
                label=y,
                categorical_feature=[c for c in cat_use if c in x.columns],
                free_raw_data=True,
            )

            try:
                booster = lgb.train(
                    local_params,
                    dtrain,
                    num_boost_round=rounds_per_chunk,
                    init_model=booster,
                    keep_training_booster=True,
                )
            except Exception as e:
                if use_gpu:
                    print(f'GPU failed, fallback CPU: {e}')
                    local_params.pop('device', None)
                    use_gpu = False
                    booster = lgb.train(
                        local_params,
                        dtrain,
                        num_boost_round=rounds_per_chunk,
                        init_model=booster,
                        keep_training_booster=True,
                    )
                else:
                    raise

            del chunk_df, part, x, y, dtrain
            gc.collect()

        if booster is None:
            raise RuntimeError('No booster trained for selected folds.')
        return booster

    def predict_for_fold(booster, cols_use: list[str], cat_cols_all: list[str], valid_fold_id: int) -> pd.DataFrame:
        labels_map = _load_labels_map()
        cat_use = [c for c in cat_cols_all if c in cols_use]
        rows = []

        for cp in train_chunk_paths:
            chunk_df = pd.read_parquet(cp)
            if chunk_df.empty:
                continue

            chunk_df['customer_ID'] = chunk_df['customer_ID'].astype('string')
            if 'target' not in chunk_df.columns:
                chunk_df = chunk_df.merge(labels_map, on='customer_ID', how='left')

            chunk_df = chunk_df[chunk_df['target'].notna()].copy()
            if chunk_df.empty:
                del chunk_df
                gc.collect()
                continue

            chunk_df['fold_id'] = add_fold_id(chunk_df, N_OOF_FOLDS)
            vp = chunk_df[chunk_df['fold_id'] == valid_fold_id].copy()
            if vp.empty:
                del chunk_df, vp
                gc.collect()
                continue

            for c in cols_use:
                if c not in vp.columns:
                    vp[c] = np.nan

            x = _prepare_lgb_frame(vp[cols_use], cat_use)
            pred = booster.predict(x)

            rows.append(pd.DataFrame({
                'customer_ID': vp['customer_ID'].values,
                'target': vp['target'].astype(int).values,
                'pred': pred,
            }))

            del chunk_df, vp, x, pred
            gc.collect()

        if not rows:
            return pd.DataFrame(columns=['customer_ID', 'target', 'pred'])
        return pd.concat(rows, axis=0, ignore_index=True)

    # Branches and params
    if MODEL_META_PATH.exists():
        _meta = json.loads(MODEL_META_PATH.read_text())
        feature_cols_all = list(_meta['feature_columns'])
        cat_cols_all = list(_meta.get('categorical_columns', []))
    elif LGB_META_PATH.exists():
        _meta = json.loads(LGB_META_PATH.read_text())
        feature_cols_all = list(_meta['feature_cols'])
        cat_cols_all = list(_meta.get('cat_cols', []))
    else:
        feature_cols_all = feature_cols[:]
        cat_cols_all = cat_cols[:]
    branch_a_cols, branch_b_cols = get_branch_cols(feature_cols_all)

    a_params = {
        'objective': 'binary', 'learning_rate': 0.03, 'num_leaves': 64,
        'feature_fraction': 0.85, 'bagging_fraction': 0.9, 'bagging_freq': 1,
        'min_data_in_leaf': 80, 'seed': 42, 'n_jobs': -1, 'verbose': -1,
    }
    b_params = {
        'objective': 'binary', 'learning_rate': 0.02, 'num_leaves': 96,
        'feature_fraction': 0.75, 'bagging_fraction': 0.85, 'bagging_freq': 1,
        'min_data_in_leaf': 120, 'seed': 77, 'n_jobs': -1, 'verbose': -1,
    }
    if GPU_AVAILABLE:
        a_params['device'] = 'gpu'
        b_params['device'] = 'gpu'

    # Full OOF over all folds
    oof_parts = []
    for f in range(N_OOF_FOLDS):
        train_folds = set(range(N_OOF_FOLDS)) - {f}
        print(f'OOF fold {f+1}/{N_OOF_FOLDS}: train_folds={sorted(train_folds)}, valid_fold={[f]}')

        m_a = fit_incremental_booster_for_folds(branch_a_cols, cat_cols_all, a_params, rounds_per_chunk=14, train_fold_ids=train_folds)
        m_b = fit_incremental_booster_for_folds(branch_b_cols, cat_cols_all, b_params, rounds_per_chunk=18, train_fold_ids=train_folds)

        va = predict_for_fold(m_a, branch_a_cols, cat_cols_all, valid_fold_id=f)
        vb = predict_for_fold(m_b, branch_b_cols, cat_cols_all, valid_fold_id=f)

        if va.empty or vb.empty:
            continue

        fold_df = va.rename(columns={'pred': 'pred_a'}).merge(
            vb[['customer_ID', 'pred']].rename(columns={'pred': 'pred_b'}),
            on='customer_ID',
            how='inner'
        )
        oof_parts.append(fold_df)

        del m_a, m_b, va, vb, fold_df
        gc.collect()

    if not oof_parts:
        raise RuntimeError('No OOF rows produced.')

    oof_df = pd.concat(oof_parts, axis=0, ignore_index=True)
    oof_df = oof_df.dropna(subset=['target'])

    # Meta model on full OOF
    meta_X = pd.DataFrame({
        'pred_a': oof_df['pred_a'].values,
        'pred_b': oof_df['pred_b'].values,
        'pred_rank_mean': 0.5 * (rank01(oof_df['pred_a'].values) + rank01(oof_df['pred_b'].values)),
    })
    meta_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
    meta_model.fit(meta_X, oof_df['target'].astype(int).values)

    meta_raw = meta_model.predict_proba(meta_X)[:, 1]
    blend_raw = 0.5 * rank01(oof_df['pred_a'].values) + 0.5 * rank01(oof_df['pred_b'].values)
    final_valid = 0.7 * rank01(meta_raw) + 0.3 * blend_raw

    score_a = amex_metric_np(oof_df['target'].values, oof_df['pred_a'].values)
    score_b = amex_metric_np(oof_df['target'].values, oof_df['pred_b'].values)
    score_meta = amex_metric_np(oof_df['target'].values, meta_raw)
    score_final = amex_metric_np(oof_df['target'].values, final_valid)
    print({'amex_a': round(score_a,6), 'amex_b': round(score_b,6), 'amex_meta': round(score_meta,6), 'amex_final': round(score_final,6)})

    # Optional CatBoost stacker on OOF branch predictions
    cb_meta_model = None
    cb_meta_path = ADV_V2_DIR / 'catboost_meta.cbm'
    if HAS_CATBOOST:
        cb_train = pd.DataFrame({
            'pred_a': oof_df['pred_a'].values,
            'pred_b': oof_df['pred_b'].values,
            'pred_rank_mean': 0.5 * (rank01(oof_df['pred_a'].values) + rank01(oof_df['pred_b'].values)),
        })
        cb_target = oof_df['target'].astype(int).values

        if cb_meta_path.exists() and (not RETRAIN_MODELS):
            cb_meta_model = cb.CatBoostClassifier()
            cb_meta_model.load_model(str(cb_meta_path))
            print('Loaded CatBoost meta model from disk.')
        else:
            cb_meta_model = cb.CatBoostClassifier(
                loss_function='Logloss',
                eval_metric='AUC',
                iterations=500,
                learning_rate=0.03,
                depth=4,
                l2_leaf_reg=8.0,
                random_seed=RANDOM_STATE,
                verbose=False,
            )
            cb_meta_model.fit(cb_train, cb_target)
            cb_meta_model.save_model(str(cb_meta_path))
            print('Saved CatBoost meta model to disk.')

        cb_oof = cb_meta_model.predict_proba(cb_train)[:, 1]
        score_cb_meta = amex_metric_np(cb_target, cb_oof)
        final_valid = 0.5 * final_valid + 0.5 * rank01(cb_oof)
        score_final = amex_metric_np(cb_target, final_valid)
        print({'amex_cb_meta': round(score_cb_meta, 6), 'amex_final_blended': round(score_final, 6)})

    # Final models trained on all folds
    m_a_final_path = ADV_V2_DIR / 'branch_a_final.pkl'
    m_b_final_path = ADV_V2_DIR / 'branch_b_final.pkl'
    meta_path = ADV_V2_DIR / 'meta_model.joblib'

    if m_a_final_path.exists() and m_b_final_path.exists() and meta_path.exists() and (not RETRAIN_MODELS):
        m_a_final = joblib.load(m_a_final_path)
        m_b_final = joblib.load(m_b_final_path)
        meta_model = joblib.load(meta_path)
        print('Loaded advanced v2 models from disk.')
    else:
        all_folds = set(range(N_OOF_FOLDS))
        m_a_final = fit_incremental_booster_for_folds(branch_a_cols, cat_cols_all, a_params, rounds_per_chunk=16, train_fold_ids=all_folds)
        m_b_final = fit_incremental_booster_for_folds(branch_b_cols, cat_cols_all, b_params, rounds_per_chunk=20, train_fold_ids=all_folds)
        joblib.dump(m_a_final, m_a_final_path)
        joblib.dump(m_b_final, m_b_final_path)
        joblib.dump(meta_model, meta_path)
        print('Saved advanced v2 models to disk.')

    # Stream test predictions
    for old in SUBMISSION_PARTS_DIR.glob('adv_v2_part_*.csv'):
        old.unlink()

    for i, cp in enumerate(tqdm(test_chunk_paths, desc='Scoring test chunks (adv v2)')):
        cdf = pd.read_parquet(cp)
        if cdf.empty:
            continue

        for c in branch_a_cols:
            if c not in cdf.columns:
                cdf[c] = np.nan
        for c in branch_b_cols:
            if c not in cdf.columns:
                cdf[c] = np.nan

        xa = _prepare_lgb_frame(cdf[branch_a_cols], [c for c in cat_cols_all if c in branch_a_cols])
        xb = _prepare_lgb_frame(cdf[branch_b_cols], [c for c in cat_cols_all if c in branch_b_cols])

        pa = np.asarray(m_a_final.predict(xa)).reshape(-1)
        pb = np.asarray(m_b_final.predict(xb)).reshape(-1)

        if len(pa) != len(cdf) or len(pb) != len(cdf):
            raise RuntimeError(f'v2 prediction length mismatch on {cp.name}: rows={len(cdf)} pa={len(pa)} pb={len(pb)}')

        if len(cdf) >= 500 and (np.unique(pa).size <= 1 or np.unique(pb).size <= 1):
            raise RuntimeError(f'v2 collapsed chunk predictions on {cp.name}.')

        meta_chunk = pd.DataFrame({
            'pred_a': pa,
            'pred_b': pb,
            'pred_rank_mean': 0.5 * (rank01(pa) + rank01(pb)),
        })
        pmeta = meta_model.predict_proba(meta_chunk)[:, 1]
        if HAS_CATBOOST and ('cb_meta_model' in globals()) and (cb_meta_model is not None):
            pcb = cb_meta_model.predict_proba(meta_chunk)[:, 1]
        else:
            pcb = pmeta

        out = pd.DataFrame({'customer_ID': cdf['customer_ID'].values, 'pred_a': pa, 'pred_b': pb, 'pred_meta': pmeta, 'pred_cb': pcb})
        out.to_csv(SUBMISSION_PARTS_DIR / f'adv_v2_part_{i:03d}.csv', index=False)

        del cdf, xa, xb, pa, pb, meta_chunk, pmeta, out
        gc.collect()

    adv_parts = pd.concat([pd.read_csv(x) for x in sorted(SUBMISSION_PARTS_DIR.glob('adv_v2_part_*.csv'))], ignore_index=True)
    base_blend = 0.7 * rank01(adv_parts['pred_meta'].values) + 0.3 * (0.5 * rank01(adv_parts['pred_a'].values) + 0.5 * rank01(adv_parts['pred_b'].values))
    if 'pred_cb' in adv_parts.columns:
        cb_blend = rank01(adv_parts['pred_cb'].values)
        adv_parts['prediction'] = 0.65 * base_blend + 0.35 * cb_blend
    else:
        adv_parts['prediction'] = base_blend

    adv_submission = adv_parts[['customer_ID', 'prediction']].copy()
    adv_submission = adv_submission.drop_duplicates(subset=['customer_ID'], keep='last')

    # Enforce Kaggle-required id set/order
    if SAMPLE_SUBMISSION_PATH.exists():
        sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        adv_submission = sample_sub[['customer_ID']].merge(adv_submission, on='customer_ID', how='left')
        if adv_submission['prediction'].isna().any():
            fill_val = adv_submission['prediction'].median(skipna=True)
            if pd.isna(fill_val):
                fill_val = 0.5
            adv_submission['prediction'] = adv_submission['prediction'].fillna(fill_val)
    else:
        print(f'Warning: sample submission file not found at {SAMPLE_SUBMISSION_PATH}. Saving raw submission order.')

    unique_pred = int(adv_submission['prediction'].nunique())
    if unique_pred < 1000:
        raise RuntimeError(f'v2 submission collapsed: only {unique_pred} unique predictions.')

    adv_submission.to_csv(ADVANCED_SUBMISSION_V2_PATH, index=False)

    print({
        'saved_to': str(ADVANCED_SUBMISSION_V2_PATH),
        'rows': len(adv_submission),
        'unique_ids': adv_submission['customer_ID'].nunique(),
        'null_predictions': int(adv_submission['prediction'].isna().sum()),
        'unique_predictions': unique_pred,
        'oof_amex_final': round(score_final, 6)
    })
    adv_submission.head()







## 14. Advanced Model v3 (Online-Inspired Meta Ensemble)

This `v3` model keeps `v2` as base and adds a second-layer meta ensemble over `v2` part predictions (`pred_a`, `pred_b`, `pred_meta`, optional `pred_cb`).

Design motivation (from high-ranking AMEX solution patterns): heterogeneous model blending + rank-based ensembling + robust stacker layer.


In [ ]:
if not should_run('v3'):
    print("Skipping advanced v3 pipeline (TRAIN_VERSION != 'v3').")
else:
    # v3 depends on v2 part-level outputs. Run v2 first in the same or previous session.
    ADVANCED_SUBMISSION_V3_PATH = SUBMISSION_LAYER_DIR / 'advanced_submission_v3.csv'
    ADV_V3_DIR = MODEL_DIR / 'advanced_v3_meta'
    ADV_V3_DIR.mkdir(parents=True, exist_ok=True)

    v2_parts = sorted(SUBMISSION_PARTS_DIR.glob('adv_v2_part_*.csv'))
    if not v2_parts:
        raise RuntimeError('No v2 part predictions found (adv_v2_part_*.csv). Run v2 pipeline first.')

    v2_df = pd.concat([pd.read_csv(x) for x in v2_parts], ignore_index=True)
    req_cols = {'customer_ID', 'pred_a', 'pred_b', 'pred_meta'}
    if not req_cols.issubset(set(v2_df.columns)):
        raise RuntimeError(f'v2 parts missing required columns: {req_cols - set(v2_df.columns)}')

    # Build meta feature matrix for final test-time blending.
    v2_df['pred_rank_mean'] = 0.5 * (rank01(v2_df['pred_a'].values) + rank01(v2_df['pred_b'].values))

    # If CatBoost branch exists from v2, include it; else use pred_meta fallback.
    if 'pred_cb' not in v2_df.columns:
        v2_df['pred_cb'] = v2_df['pred_meta']

    # Calibrated rank blend inspired by top-solution ensemble behavior.
    # This keeps ranking robustness and adds diversity beyond v2's fixed blend.
    score_components = {
        'meta_rank': rank01(v2_df['pred_meta'].values),
        'cb_rank': rank01(v2_df['pred_cb'].values),
        'a_rank': rank01(v2_df['pred_a'].values),
        'b_rank': rank01(v2_df['pred_b'].values),
        'rank_mean': rank01(v2_df['pred_rank_mean'].values),
    }

    # Hand-tuned robust weights (sum to 1.0), chosen to improve stability vs single branch dominance.
    w = {
        'meta_rank': 0.38,
        'cb_rank': 0.24,
        'a_rank': 0.14,
        'b_rank': 0.14,
        'rank_mean': 0.10,
    }

    final_pred = np.zeros(len(v2_df), dtype=float)
    for k, arr in score_components.items():
        final_pred += w[k] * arr

    v3_submission = v2_df[['customer_ID']].copy()
    v3_submission['prediction'] = final_pred
    v3_submission = v3_submission.drop_duplicates(subset=['customer_ID'], keep='last')

    # Enforce Kaggle-required id set/order.
    if SAMPLE_SUBMISSION_PATH.exists():
        sample_sub = pd.read_csv(SAMPLE_SUBMISSION_PATH)
        v3_submission = sample_sub[['customer_ID']].merge(v3_submission, on='customer_ID', how='left')
        if v3_submission['prediction'].isna().any():
            fill_val = v3_submission['prediction'].median(skipna=True)
            if pd.isna(fill_val):
                fill_val = 0.5
            v3_submission['prediction'] = v3_submission['prediction'].fillna(fill_val)

    unique_pred = int(v3_submission['prediction'].nunique())
    if unique_pred < 1000:
        raise RuntimeError(f'v3 submission collapsed: only {unique_pred} unique predictions.')

    v3_submission.to_csv(ADVANCED_SUBMISSION_V3_PATH, index=False)

    print({
        'saved_to': str(ADVANCED_SUBMISSION_V3_PATH),
        'rows': int(len(v3_submission)),
        'unique_ids': int(v3_submission['customer_ID'].nunique()),
        'null_predictions': int(v3_submission['prediction'].isna().sum()),
        'unique_predictions': unique_pred,
        'weights': w,
    })
    v3_submission.head()
